# 4. Mushroom foraging

The [mushroom dataset](https://www.kaggle.com/datasets/dhinaharp/mushroom-dataset) contains data about approximately 60000 mushrooms, and your task is to classify them as either edible or poisonous. You can read about the features [here](https://www.kaggle.com/datasets/uciml/mushroom-classification) and import the data using:

In [4]:
# Import Statements
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [24]:
# Data prep
pd.set_option('display.max_columns', 1000)
df = pd.read_csv('secondary_data.csv', delimiter = ';')
df.head()

## Before the feature leakage analysis was performed, both linear regressions and random forest would reach 100% in all evaluation metrics, as there were features that could 100% give away the edibility of a mushroom based on that feature alone, they are removed in the leakage analysis below.

# === Feature Leakage Analysis ===
# Identify and print columns with perfect or near-perfect class separation
print("\nChecking for class leakage in categorical features:")
leaky_features = []
for col in df.select_dtypes(include='object').columns.drop('class'):
    distribution = df.groupby(col)['class'].value_counts(normalize=True).unstack().fillna(0)
    for value in distribution.index:
        if (distribution.loc[value] == 1.0).any():
            print(f"'{col}' contains a value '{value}' that perfectly predicts class")
            leaky_features.append(col)
            break

# Now drop columns with excessive missing data or class leakage
# Print leaky features and columns used to confirm correct preprocessing
print("Leaky features detected:", leaky_features)
cols_to_drop = list(set(leaky_features))  # Only use dynamically detected leaky features

df = df.drop(columns=cols_to_drop)


# Drop rows with remaining missing values
df = df.dropna()


Checking for class leakage in categorical features:
'stem-root' contains a value 'c' that perfectly predicts class
'stem-surface' contains a value 'f' that perfectly predicts class
'stem-color' contains a value 'b' that perfectly predicts class
'veil-color' contains a value 'e' that perfectly predicts class
'ring-type' contains a value 'm' that perfectly predicts class
'spore-print-color' contains a value 'g' that perfectly predicts class
'habitat' contains a value 'p' that perfectly predicts class
Leaky features detected: ['stem-root', 'stem-surface', 'stem-color', 'veil-color', 'ring-type', 'spore-print-color', 'habitat']


It's up to you how you approach this data, but at a minimum, your analysis should include:

* Informed **data preparation**.
* 2 different classification models, one of which must be **logistic regression**.
* A discussion of which **performance metric** is most relevant for the evaluation of your models.
* 2 different **validation methodologies** used to tune hyperparameters.
* **Confusion matrices** for your models, and associated comments.

In [25]:
# 2 Different classification models

# Separate features and target
y = df['class']
X = df.drop(columns=['class'])

# Encode categorical variables
le = LabelEncoder()
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = le.fit_transform(X[col])

# Encode target variable
# Use a separate LabelEncoder for the target to avoid overwriting the feature encoder (le),
# and to preserve class label mapping for interpretation
le_class = LabelEncoder()
y = le_class.fit_transform(y)

# Print final features used in model for verification
print("Features used in model:", X.columns.tolist())

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Model 1: Logistic Regression ---
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)

print("\nLogistic Regression Report:")
print(classification_report(y_test, y_pred_log))
print("Accuracy:", accuracy_score(y_test, y_pred_log))

# --- Model 2: Random Forest Classifier ---
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\nRandom Forest Classifier Report:")
print(classification_report(y_test, y_pred_rf))
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

Features used in model: ['cap-diameter', 'cap-shape', 'cap-surface', 'cap-color', 'does-bruise-or-bleed', 'gill-attachment', 'gill-spacing', 'gill-color', 'stem-height', 'stem-width', 'veil-type', 'has-ring', 'season']


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)

# === Discussion: Evaluating Model Performance ===
In this classification task, we evaluate models using multiple metrics:
- Precision: how many predicted positives are actually correct
- Recall: how many actual positives were correctly predicted
- F1 Score: harmonic mean of precision and recall

Since misclassifying a poisonous mushroom as edible could be dangerous,
recall is especially important for the poisonous class (assumed to be label 1, based on alphabetical encoding).
However, high precision is also critical to avoid false alarms.

F1 score provides a balanced view between precision and recall,
especially when class distribution is not perfectly balanced.
Therefore, F1 score is arguably the most relevant single metric
to compare model performance in this context.